Step 1: Imports and Configuration

In [2]:
import requests
import pandas as pd
import json
import time

from pathlib import Path
from datetime import datetime, timezone

BASE_URL = "https://api.ods.od.nih.gov/dsld/v9"

SUPPLEMENTS = [
    "Magnesium",
    "Vitamin D",
    "Ashwagandha",
    "Melatonin",
    "Omega-3",
    "Vitamin B12",
    "Iron",
    "Turmeric",
    "Probiotics",
    "Creatine"
]

TARGET_PER_CATEGORY = 40

# Resolve the project root
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "dsld"
RAW_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()

Step 2 : Function to Search DSLD products 

In [3]:
def search_dsld(supplements, size=40, session=None):
    """
    Search the DSLD database for multiple supplement terms.

    Parameters
    ----------
    supplements : list[str]
        List of supplement search terms.
    size : int
        Maximum search results per supplement.
    session : requests.Session
        HTTP session.

    Returns
    -------
    search_results : dict
        Raw API search responses by supplement.
    all_ids : dict
        Mapping of unique DSLD IDs to matched search terms.
    """

    if session is None:
        session = requests.Session()

    search_results = {}
    all_ids = {}

    for supplement in supplements:

        response = session.get(
            f"{BASE_URL}/search-filter",
            params={
                "q": supplement,
                "from": 0,
                "size": size
            },
            timeout=30
        )

        response.raise_for_status()
        result = response.json()

        search_results[supplement] = result

        ids = [
            str(hit["_id"])
            for hit in result.get("hits", [])
        ]

        for dsld_id in ids:
            all_ids.setdefault(dsld_id, set()).add(supplement)

        print(
            f"{supplement}: {len(ids)} retrieved "
            f"| Total matches: {result.get('stats', {}).get('count')}"
        )

        time.sleep(0.3)

    print(f"\nTotal unique DSLD IDs: {len(all_ids)}")

    return search_results, all_ids

Step 3: Fetch Complete DSLD labels

In [4]:
def fetch_dsld_labels(dsld_ids, session=None, delay=0.3):
    """
    Retrieve complete DSLD product labels.

    Parameters
    ----------
    dsld_ids : iterable
        Collection of DSLD record identifiers.
    session : requests.Session
        HTTP session.
    delay : float
        Delay between requests in seconds.

    Returns
    -------
    labels : list
        Complete product-label records.
    failed_ids : list
        Failed requests and associated errors.
    """

    if session is None:
        session = requests.Session()

    labels = []
    failed_ids = []

    dsld_ids = list(dsld_ids)

    for i, dsld_id in enumerate(dsld_ids, start=1):

        try:
            response = session.get(
                f"{BASE_URL}/label/{dsld_id}",
                timeout=30
            )

            response.raise_for_status()
            label = response.json()

            labels.append(label)

        except (requests.RequestException, ValueError) as e:
            failed_ids.append({
                "dsld_id": str(dsld_id),
                "error": str(e)
            })

        if i % 20 == 0:
            print(f"Processed {i}/{len(dsld_ids)} labels")

        time.sleep(delay)

    print(f"\nSuccessfully downloaded: {len(labels)}")
    print(f"Failed downloads: {len(failed_ids)}")

    return labels, failed_ids

Step 4 : Function to save data 

In [5]:
def save_dsld_data(
    labels,
    search_results,
    all_ids,
    failed_ids,
    output_dir,
    requested_per_query=None
):
    """
    Save raw DSLD records and collection metadata.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Save complete label records
    labels_dir = output_dir / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)
    for label in labels:

        dsld_id = label["id"]
        file_path = labels_dir / f"label_{dsld_id}.json"

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(label, f, indent=2, ensure_ascii=False)

    # 2. Save original search responses
    with open(output_dir / "search_results.json", "w", encoding="utf-8") as f:
        json.dump(search_results, f, indent=2, ensure_ascii=False)

    # 3. Save collection metadata
    manifest = {
        "source": "NIH Dietary Supplement Label Database",
        "api_version": "v9",
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "queries": list(search_results.keys()),
        "requested_per_query": requested_per_query,
        "unique_ids_found": len(all_ids),
        "labels_downloaded": len(labels),
        "failed_requests": failed_ids,
        "category_matches": {
            str(dsld_id): sorted(categories)
            for dsld_id, categories in all_ids.items()
        }
    }

    with open(output_dir / "collection_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(labels)} labels to {labels_dir.resolve()}")

    return manifest

Step 5: Execute data Collection pipeline

In [6]:
# Step 1: Search for supplement products
search_results, all_ids = search_dsld(
    supplements=SUPPLEMENTS,
    size=TARGET_PER_CATEGORY,
    session=session
)

# Step 2: Retrieve complete labels
labels, failed_ids = fetch_dsld_labels(
    dsld_ids=all_ids.keys(),
    session=session
)

# Step 3: Save the collected data
manifest = save_dsld_data(
    labels=labels,
    search_results=search_results,
    all_ids=all_ids,
    failed_ids=failed_ids,
    output_dir=RAW_DIR,
    requested_per_query=TARGET_PER_CATEGORY
)

Magnesium: 40 retrieved | Total matches: 77199
Vitamin D: 40 retrieved | Total matches: 77594
Ashwagandha: 40 retrieved | Total matches: 3772
Melatonin: 40 retrieved | Total matches: 3144
Omega-3: 40 retrieved | Total matches: 13269
Vitamin B12: 40 retrieved | Total matches: 64710
Iron: 40 retrieved | Total matches: 26154
Turmeric: 40 retrieved | Total matches: 8123
Probiotics: 40 retrieved | Total matches: 6643
Creatine: 40 retrieved | Total matches: 3957

Total unique DSLD IDs: 400
Processed 20/400 labels
Processed 40/400 labels
Processed 60/400 labels
Processed 80/400 labels
Processed 100/400 labels
Processed 120/400 labels
Processed 140/400 labels
Processed 160/400 labels
Processed 180/400 labels
Processed 200/400 labels
Processed 220/400 labels
Processed 240/400 labels
Processed 260/400 labels
Processed 280/400 labels
Processed 300/400 labels
Processed 320/400 labels
Processed 340/400 labels
Processed 360/400 labels
Processed 380/400 labels
Processed 400/400 labels

Successfully d

Step 6 : Validate records 

In [7]:
def summarize_dsld_labels(labels):

    rows = []

    for label in labels:
        rows.append({
            "dsld_id": label.get("id"),
            "product_name": label.get("fullName"),
            "brand": label.get("brandName"),
            "ingredient_count": len(label.get("ingredientRows") or []),
            "statement_count": len(label.get("statements") or []),
            "serving_size_count": len(label.get("servingSizes") or [])
        })

    df = pd.DataFrame(rows)

    print("Total labels:", len(df))
    print("Unique DSLD IDs:", df["dsld_id"].nunique())
    print("\nMissing values:")
    print(df.isna().sum())

    return df


df_summary = summarize_dsld_labels(labels)

display(df_summary.head(10))

Total labels: 400
Unique DSLD IDs: 400

Missing values:
dsld_id               0
product_name          0
brand                 0
ingredient_count      0
statement_count       0
serving_size_count    0
dtype: int64


,dsld_id,product_name,brand,ingredient_count,statement_count,serving_size_count
0,19279,Magnesium,Nutra Sport,2,10,1
1,242165,Quad-Magnesium,VitaMonk,1,14,1
2,183121,Magnesium 3,InterPlexus,1,12,1
3,204235,Magnesium Citrate,Bluebonnet,1,14,1
4,203860,Magnesium Citrate,Bluebonnet,1,14,1
5,47753,Magnesium Citrate,Physio Vites,1,15,1
6,21158,Magnesium Citrate,Vitamin World,1,17,1
7,184238,Magnesium Citrate,Allergy Research Group,1,10,1
8,7841,Magnesium Optimizer,Jarrow Formulas,3,15,1
9,29547,Magnesium Optimizer,Jarrow Formulas,3,17,1


In [8]:
print("Labels with no ingredient rows:",
      (df_summary["ingredient_count"] == 0).sum())

print("Labels with no statements:",
      (df_summary["statement_count"] == 0).sum())

print("Labels with no serving sizes:",
      (df_summary["serving_size_count"] == 0).sum())

print("\nTotal ingredient rows:",
      df_summary["ingredient_count"].sum())

print("Total label statements:",
      df_summary["statement_count"].sum())

Labels with no ingredient rows: 0
Labels with no statements: 0
Labels with no serving sizes: 0

Total ingredient rows: 1553
Total label statements: 6462
